# Pruebas unitarias y métricas

Notebook para ejecutar paso a paso tres pruebas pequeñas sobre el planificador y una prueba del benchmark existente. Las pruebas llaman al código JavaScript del proyecto, sin duplicar su lógica ni instalar dependencias nuevas.

Ejecuta primero la celda de preparación y luego las pruebas una por una. El benchmark puede tardar más que las pruebas pequeñas.

## 1. Importar librerías necesarias

Usaremos `unittest` y `subprocess` para comprobar los módulos JavaScript existentes desde Python.

In [ ]:
import json
import subprocess
import textwrap
import unittest
from pathlib import Path

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "package.json").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Abre el notebook desde una carpeta dentro del repositorio.")


def ejecutar_node(codigo):
    resultado = subprocess.run(
        ["node", "-e", textwrap.dedent(codigo)],
        cwd=ROOT,
        capture_output=True,
        text=True,
    )
    if resultado.returncode != 0:
        raise AssertionError(resultado.stderr or resultado.stdout)
    print(resultado.stdout.strip())

print(f"Repositorio: {ROOT}")

## 2. Definir funciones a probar

Las comprobaciones llaman a `GenerarFrecuencias`, `GrafoHorarios` y `Planificador` directamente. Para métricas se ejecuta el benchmark ya existente en `scripts/benchmark.js`.

In [ ]:
class PruebasPlanificador(unittest.TestCase):
    def test_frecuencias_respetan_dias_y_paradas(self):
        ejecutar_node("""
            const assert = require('node:assert/strict');
            const { generarFrecuencias } = require('./src/main/domain/datos/GenerarFrecuencias');
            const ruta = {
              id: 'prueba', cooperativa: 'Demo', nombre: 'Loja-Cariamanga',
              dias: [1], salidas: ['08:00'],
              paradas: [
                { terminal: 'Loja', minutos: 0 },
                { terminal: 'Catamayo', minutos: 30 },
                { terminal: 'Cariamanga', minutos: 60 },
              ],
            };
            const { frecuencias, errores } = generarFrecuencias([ruta], ['2026-11-02']);
            assert.equal(errores.length, 0);
            assert.equal(frecuencias.length, 3);
            assert.equal(Date.parse(frecuencias[0].llegada) - Date.parse(frecuencias[0].salida), 30 * 60000);
            console.log('Frecuencias y días: OK');
        """)

    def test_grafo_descarta_viaje_invalido(self):
        ejecutar_node("""
            const assert = require('node:assert/strict');
            const GrafoHorarios = require('./src/main/domain/graph/GrafoHorarios');
            const grafo = new GrafoHorarios().cargar([
              { id: 'valida', origen: 'Loja', destino: 'Catamayo', salida: '2026-11-02T13:00:00Z', llegada: '2026-11-02T14:00:00Z' },
              { id: 'invalida', origen: 'Loja', destino: 'Catamayo', salida: '2026-11-02T13:00:00Z', llegada: '2026-11-02T12:00:00Z' },
            ]);
            assert.equal(grafo.errores.length, 1);
            assert.equal(grafo.nodos.length, 2);
            assert.deepEqual(grafo.aristas.map((arista) => arista.tipo), ['viaje']);
            console.log('Validación del grafo: OK');
        """)

    def test_planificador_encuentra_ruta(self):
        ejecutar_node("""
            const assert = require('node:assert/strict');
            const path = require('node:path');
            const { crearPlanificador } = require('./src/main/services/planificadorService');
            const desde = '2026-11-02T10:30:00-05:00';
            const planificador = crearPlanificador(path.join(process.cwd(), 'src', 'main'), desde);
            const ruta = planificador.buscarRuta({ origen: 'Loja', destino: 'Cariamanga', desde, criterio: 'tiempo' });
            assert.ok(ruta);
            assert.ok(ruta.tramos.length > 0);
            assert.equal(ruta.tramos[0].desde, 'Loja');
            assert.equal(ruta.criterio, 'tiempo');
            console.log('Búsqueda de ruta: OK');
        """)

    def test_benchmark_existente(self):
        resultado = subprocess.run(
            ['node', 'scripts/benchmark.js'],
            cwd=ROOT,
            capture_output=True,
            text=True,
            check=True,
        )
        metricas = json.loads(resultado.stdout)
        self.assertIn('real', metricas)
        self.assertIn('escala', metricas)
        self.assertGreater(len(metricas['escala']), 0)
        print(f"Benchmark: OK ({len(metricas['escala'])} tamaños medidos)")

## 3. Crear pruebas unitarias básicas

La clase `PruebasPlanificador` define tres casos pequeños y uno para comprobar que el benchmark existente entrega sus métricas.

## 4. Ejecutar pruebas y mostrar resultados

Ejecuta cada celda de abajo en orden. Cada una corre un único método de prueba y muestra el resultado; deja el benchmark para el final.

In [ ]:
suite = unittest.TestSuite([PruebasPlanificador("test_frecuencias_respetan_dias_y_paradas")])
resultado = unittest.TextTestRunner(verbosity=2).run(suite)
assert resultado.wasSuccessful(), "Falló la prueba de frecuencias"

In [ ]:
suite = unittest.TestSuite([PruebasPlanificador("test_grafo_descarta_viaje_invalido")])
resultado = unittest.TextTestRunner(verbosity=2).run(suite)
assert resultado.wasSuccessful(), "Falló la prueba del grafo"

In [ ]:
suite = unittest.TestSuite([PruebasPlanificador("test_planificador_encuentra_ruta")])
resultado = unittest.TextTestRunner(verbosity=2).run(suite)
assert resultado.wasSuccessful(), "Falló la prueba del planificador"

In [ ]:
suite = unittest.TestSuite([PruebasPlanificador("test_benchmark_existente")])
resultado = unittest.TextTestRunner(verbosity=2).run(suite)
assert resultado.wasSuccessful(), "Falló el benchmark"